# Polar splines in the Struphy de Rham complex

Domains with a magnetic axis (cylinders, tori, stellarators) are parametrized by a mapping $F$ that collapses the
logical boundary $\eta_1 = 0$ onto a line, the *pole*. Tensor-product splines pushed forward by $F$ are in general
discontinuous there. Struphy's **polar splines** are the subspaces of the tensor-product spaces that are smooth at the
pole and still form a de Rham complex with commuting projectors. They are selected in the `DerhamOptions`:

```python
DerhamOptions(polar_splines=True, polar_regularity=r)
```

For regularity $\mathfrak r$ the 0-forms are $C^{\mathfrak r}$, the 1-forms $C^{\mathfrak r - 1}$ and the 2-forms
$C^{\mathfrak r - 2}$ at the pole. The construction (R. Remmerswaal, homotopy operators; see the developer tutorial
*Homotopy operators and high-order polar complexes*) works for every domain with a pole and every regularity
$\mathfrak r \leq p_1$ with $N_2 \geq 2\mathfrak r + 1$. **Regularity 1 spans exactly the former C1 polar splines**
(Toshniwal et al. 2017; Holderied & Possanner 2022).

This tutorial shows

1. domains with a pole,
2. the polar spaces of different regularities,
3. why regularity 1 is the C1 construction,
4. the smoothness of polar splines at the pole,
5. the commuting projectors.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from struphy.feec.psydac_derham import Derham
from struphy.geometry import domains
from struphy.io.options import DerhamOptions
from struphy.topology.grids import TensorProductGrid


def polar_derham(domain, regularity, num_elements=(8, 24, 1), degree=(5, 5, 1), polar=True):
    # a 2D complex (one cell in eta3) on the cross-section of the domain
    options = DerhamOptions(
        degree=degree,
        bcs=(("free", "free"), None, None),
        polar_splines=polar,
        polar_regularity=regularity,
        nquads_proj=tuple(p + 3 for p in degree),
    )
    return Derham(TensorProductGrid(num_elements=num_elements), options, domain=domain)

## 1. Domains with a pole

Every Struphy domain whose first logical boundary collapses to a line can be used: analytic mappings such as
`HollowCylinder(a1=0)` and `HollowTorus(a1=0)`, the star-shaped `PerturbedDiskCylinder`, the `RoundedSquareCylinder`,
the helical `HelicalTube` and spline mappings such as `IGAPolarCylinder`, `IGAPolarTorus` or `Tokamak`. Unlike the
former C1 construction, the polar splines do not need a spline mapping with the resolution of the grid, and the number
of angular cells is arbitrary.

In [ ]:
for domain in (
    domains.HollowCylinder(a1=0.0, a2=1.0, Lz=1.0),
    domains.PerturbedDiskCylinder(R1=1.0, eps=0.2, k=3, Lz=1.0),
    domains.RoundedSquareCylinder(),
):
    domain.show(figsize=(9, 4))

## 2. Polar spaces of regularity $\mathfrak r$

The polar spline spaces keep the tensor-product splines of the outer rings and replace the inner rings by
$(\mathfrak r + 1)(\mathfrak r + 2)/2$ projected polynomials of degree $\mathfrak r$ (for 0-forms; similarly for 1- and
2-forms). In the Struphy data structures (`PolarVector`), the first `n_rings` rings of each component hold the
`n_polar` polar coefficients and the remaining rings are tensor-product coefficients:

In [ ]:
domain = domains.PerturbedDiskCylinder(R1=1.0, eps=0.2, k=3, Lz=1.0)
print("r   space   dimension   n_polar        n_rings")
for r in (1, 2, 3):
    derham = polar_derham(domain, r)
    for space_id in ("H1", "Hcurl", "Hdiv", "L2"):
        space = derham.polar_coeff_spaces[derham.space_to_form[space_id]]
        print(f"{r}   {space_id:6s}  {space.dimension:9d}   {str(space.n_polar):13s}  {space.n_rings}")

## 3. Regularity 1 is the C1 construction

A tensor-product 0-form is $C^1$ at the pole exactly when, on the two innermost rings, its coefficients are those of an
affine function $a + b x + c y$ (Toshniwal et al. 2017). On a spline mapping these coefficients are $a + b\,c^x_{ij} + c\,c^y_{ij}$
with the control points $c_{ij}$ of the mapping, which is how the former C1 basis was built. The polar 0-forms of
regularity 1 span the same space: on the rings 0 and 1, they lie in the span of the projections of $1$, $x$ and $y$.
(The test `struphy/polar/tests/test_polar_regularity_one.py` compares all degrees with the former C1 extraction
matrices.)

In [ ]:
domain = domains.IGAPolarCylinder(a=1.0, Lz=1.0, num_elements=(8, 12), degree=(3, 3))
derham = polar_derham(domain, 1, num_elements=(8, 12, 1), degree=(3, 3, 1))
tensor = polar_derham(domain, 1, num_elements=(8, 12, 1), degree=(3, 3, 1), polar=False)
n2 = 12

affine = [tensor.P0(lambda e1, e2, e3: 1.0 + 0 * e1)]
for k in range(2):
    affine.append(tensor.P0(lambda e1, e2, e3, k=k: domain(e1, e2, e3)[k]))
Q = np.array([v.toarray().reshape(-1, n2)[:2].ravel() for v in affine])  # rings 0 and 1 of 1, x, y

E = derham.extraction_ops["0"].blocks_ten_to_pol[0][0].toarray()[:, : 2 * n2]  # polar 0-forms on rings 0 and 1
print("rank of the affine functions on the rings 0, 1:            ", np.linalg.matrix_rank(Q))
print("rank of the polar 0-forms of regularity 1 on the rings 0, 1:", np.linalg.matrix_rank(E, tol=1e-9))
print("joint rank:                                                ", np.linalg.matrix_rank(np.vstack([Q, E]), tol=1e-9))

## 4. Smoothness at the pole

A random element of the polar space of regularity $\mathfrak r$ is $C^{\mathfrak r}$ at the pole, while a random
tensor-product spline is not even continuous. We draw random coefficients and measure the jumps of the derivatives
$d^k u / dx^k$ across the pole along the line through the rays $\eta_2 = 0$ and $\eta_2 = 1/2$ (exact one-sided
derivatives from the polynomial pieces of the first radial cell, degree-6 splines):

In [ ]:
from struphy.polar.homotopy.demos.regularity import line_derivative_jumps

domain = domains.PerturbedDiskCylinder(R1=1.0, eps=0.2, k=3, Lz=1.0)
rng = np.random.default_rng(1)


def random_field(derham):
    u = derham.create_spline_function("u", "H1")
    v = u.vector
    if hasattr(v, "pol"):
        v.pol = [rng.standard_normal(p.shape) for p in v.pol]
        v.tp._data[...] = rng.standard_normal(v.tp._data.shape)
        v.set_tp_coeffs_to_zero()
    else:
        v._data[...] = rng.standard_normal(v._data.shape)
    v.update_ghost_regions()
    u.vector = v
    return u


fields = {"tensor product": random_field(polar_derham(domain, 1, degree=(6, 6, 1), polar=False))}
for r in (1, 2, 3):
    fields[f"r = {r}"] = random_field(polar_derham(domain, r, degree=(6, 6, 1)))

print(f"{'':15s}" + "".join(f"   jump d^{k}u" for k in range(5)))
for name, u in fields.items():
    print(f"{name:15s}" + "".join(f"{j:12.1e}" for j in line_derivative_jumps(u, domain, num_cells=8, degree=6)))

The kinks are visible along the line $y = 0$ through the pole:

In [ ]:
e1 = np.linspace(0, 0.3, 301)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, u in fields.items():
    xs, values = [], []
    for e2, sign in ((0.5, -1), (0.0, 1)):
        order = slice(None, None, -1) if sign < 0 else slice(None)
        x = np.asarray(domain(e1, np.array([e2]), np.array([0.0]))[0]).ravel()
        xs.append(x[order])
        values.append(np.asarray(u(e1, np.array([e2]), np.array([0.0]))).ravel()[order])
    x, v = np.concatenate(xs), np.concatenate(values)
    v = v - np.interp(0.0, x, v)
    axes[0].plot(x, v, label=name)
    axes[1].plot(0.5 * (x[1:] + x[:-1]), np.diff(v) / np.diff(x), label=name)
axes[0].set_title("random polar spline along $y = 0$")
axes[1].set_title("its derivative $du/dx$")
for ax in axes:
    ax.axvline(0.0, color="gray", lw=0.5)
    ax.set_xlabel("$x$")
    ax.legend(fontsize=8)
fig.tight_layout()

**Exactness.** The polynomials of degree $\mathfrak r$ are reproduced exactly, and the spaces are exactly
$C^{\mathfrak r}$ in every direction, when the pullback of these polynomials lies in the logical spline space. On the
disk $R = \eta_1$ this needs radial degree $p_1 \geq \mathfrak r$, on the star-shaped disk $p_1 \geq 2\mathfrak r$, and
in both cases the angular harmonics $\cos(m\theta)$, $\sin(m\theta)$, $m \leq \mathfrak r$, which B-splines in $\eta_2$
only approximate. Along a single line the angular approximation does not enter; in all directions it does, and the
remaining non-polynomial angular harmonics $\delta_k$ of the one-sided derivatives $d^k u(0^+; \alpha)$ on 48 rays
(the measure of the talk) are of the size of the angular approximation error:

In [ ]:
from struphy.polar.homotopy.demos.regularity import field_smoothness_defects

print(f"{'':15s}   delta_1   delta_2   delta_3")
for name, u in fields.items():
    print(f"{name:15s}" + "".join(f"{d:10.1e}" for d in field_smoothness_defects(u, domain)))

## 5. Commuting projectors

The commuting projectors of the polar complex are the tensor-product projections followed by the polar dual
coefficients, $\bar\pi = \tilde F^\top \pi$. They commute with the polar derivatives, here for a function given in
logical coordinates (up to the finite differences used for its gradient):

In [ ]:
derham = polar_derham(domain, 3)
h = 1e-6


def g(a, b, c):
    return np.sin(2 * np.pi * b) * a**3 + a**4 * np.cos(4 * np.pi * b) + 0 * c


def d(fun, i):
    def derivative(a, b, c):
        plus, minus = [a, b, c], [a, b, c]
        plus[i] = plus[i] + h
        minus[i] = minus[i] - h
        return (fun(*plus) - fun(*minus)) / (2 * h)

    return derivative


u = derham.P0(g)
grad_g = derham.P1([d(g, 0), d(g, 1), d(g, 2)])
residual = derham.grad.dot(u) - grad_g
print("|| grad P0 g - P1 grad g || / || P1 grad g || =", np.sqrt(residual.inner(residual) / grad_g.inner(grad_g)))

## Further reading

* Examples with polar splines: `examples/Poisson/polar_rounded_square`, `examples/CurlCurl/polar_rounded_square`,
  `examples/ScalarWave/gaussian_perturbed_disk`, `examples/Maxwell/gaussian_perturbed_disk`,
  `examples/Maxwell/gaussian_helical_tube` and `examples/GuidingCenter/vacuum_near_axis`, and the on-axis cylinders
  and tori of `examples/Poisson`, `examples/CurlCurl` and `examples/GyrokineticPoisson`.
* *High-order polar splines* (`tutorial_polar_splines`): the polar bases and the wave equations of the talk.
* *Homotopy operators and high-order polar complexes* (`dev_tutorial_polar_homotopy`): the construction.